# KinoCompass – Mult-VAE на GPU

Mult-VAE (Liang et al., 2018) – вариационный автокодировщик: набор оценённых человеком фильмов сжимается в распределение «вкуса», декодировщик возвращает баллы всем ~21 тыс. фильмов-кандидатов. Сетка: размер «вкуса» 200 и 400, верхняя граница β 0,2 и 0,5, по 40 эпох обучения.

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`, `protocol.py`, `metrics.py`): те же люди, те же сценарии, те же метрики. Набор MovieLens 32M скачивается в память Colab.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `vae_<период>.zip` с отчётом – его нужно положить в `model/reports/`.

Время на T4 (бесплатный Colab): загрузка набора и сценарии – около 3 минут, обучение четырёх вариантов – 68 минут (эпоха – около 25 с, узкое место – сборка плотных пачек на процессоре), проверка – 3,5 минуты; всего 74 минуты. Вкладку не закрывать до конца.

In [1]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [2]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [3]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

/content
c888871 (grafted, HEAD -> main, origin/main, origin/HEAD) Добавил Mult-VAE и ноутбук Colab для него
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [4]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py vae

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
кандидаты: candidates, 21249 фильмов
сценарии: {'cold5': {'people': 3912, 'median_inputs': 5, 'median_targets': 44}, 'cold10': {'people': 3903, 'median_inputs': 10, 'median_targets': 41}, 'warm': {'people': 5677, 'median_inputs': 308, 'median_targets': 8}}
cold5 popular_year ndcg@10 0.34536
cold10 popular_year ndcg@10 0.30037
warm popular_year ndcg@10 0.08778
vae_z200_b0.2: эпоха 1/40, потеря 1217.442, 26.8 с
vae_z200_b0.2: эпоха 10/40, потеря 1106.582, 26.2 с
vae_z200_b0.2: эпоха 20/40, потеря 1102.113, 26.2 с
vae_z200_b0.2: эпоха 30/40, потеря 1097.772, 25.2 с
vae_z200_b0.2: эпоха 40/40, потеря 1094.967, 25.2 с
cold5 vae_z200_b0.2 ndcg@10 0.34573
cold10 vae_z200_b0.2 ndcg@10 0.

In [5]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/vae_{PERIOD}.zip reports/vae && ls -la reports/vae
files.download(f"/content/vae_{PERIOD}.zip")

total 168
drwxr-xr-x 2 root root   4096 Nov 11 09:08 .
drwxr-xr-x 9 root root   4096 Nov 11 09:08 ..
-rw-r--r-- 1 root root  10662 Nov 11 09:08 validation.json
-rw-r--r-- 1 root root 148813 Nov 11 09:08 validation_users.csv.gz
